# U.S. Medical Insurance Costs

In [3]:
import csv

records = []

with open("insurance.csv", newline="", encoding="utf-8-sig") as file:
    reader = csv.DictReader(file)

    for row in reader:
        records.append({
            "age": int(row["age"]),
            "sex": row["sex"],
            "bmi": float(row["bmi"]),
            "children": int(row["children"]),
            "smoker": row["smoker"],
            "region": row["region"],
            "charges": float(row["charges"]),
        })

print(f"Records loaded: {len(records):,}")
print("First record:", records[0])
print("Smoking labels:", sorted({row["smoker"] for row in records}))
print("Regions:", sorted({row["region"] for row in records}))

Records loaded: 1,338
First record: {'age': 19, 'sex': 'female', 'bmi': 27.9, 'children': 0, 'smoker': 'yes', 'region': 'southwest', 'charges': 16884.924}
Smoking labels: ['no', 'yes']
Regions: ['northeast', 'northwest', 'southeast', 'southwest']


In [4]:
from collections import defaultdict
from statistics import mean, median

def summarize_charges(records, group_key):
    groups = defaultdict(list)

    for row in records:
        groups[group_key(row)].append(row["charges"])

    for group, charges in sorted(groups.items()):
        print(
            f"{group} | n={len(charges):,} | "
            f"mean=${mean(charges):,.2f} | "
            f"median=${median(charges):,.2f}"
        )

print(f"Average age: {mean(row['age'] for row in records):.1f} years")

ages_with_children = [row['age'] for row in records if row['children'] >= 1]
if ages_with_children:
    print(f"Average age with a least one child: {mean(ages_with_children):.1f}")

print("\nCharges by region")
summarize_charges(records, lambda row: row["region"])

print("\nCharges by smoking status")
summarize_charges(records, lambda row: row["smoker"])

Average age: 39.2 years
Average age with a least one child: 39.8

Charges by region
northeast | n=324 | mean=$13,406.38 | median=$10,057.65
northwest | n=325 | mean=$12,417.58 | median=$8,965.80
southeast | n=364 | mean=$14,735.41 | median=$9,294.13
southwest | n=325 | mean=$12,346.94 | median=$8,798.59

Charges by smoking status
no | n=1,064 | mean=$8,434.27 | median=$7,345.41
yes | n=274 | mean=$32,050.23 | median=$34,456.35


Compare smokers and nonsmokers within age groups 

In [5]:
def age_group(age):
    if age < 30:
        return "18-29"
    if age < 45:
        return "30-44"
    if age < 60:
        return "45-59"
    return "60+"

def age_and_smoking_group(row):
    return age_group(row["age"]), row["smoker"]


print("Charges by age group and smoking status")
summarize_charges(records, age_and_smoking_group)

Charges by age group and smoking status
('18-29', 'no') | n=331 | mean=$4,418.57 | median=$2,709.11
('18-29', 'yes') | n=86 | mean=$27,518.04 | median=$33,604.25
('30-44', 'no') | n=301 | mean=$6,945.33 | median=$6,250.44
('30-44', 'yes') | n=91 | mean=$30,833.99 | median=$35,491.64
('45-59', 'no') | n=345 | mean=$11,871.67 | median=$10,601.41
('45-59', 'yes') | n=70 | mean=$35,889.87 | median=$40,847.36
('60+', 'no') | n=87 | mean=$15,232.71 | median=$13,831.12
('60+', 'yes') | n=27 | mean=$40,630.70 | median=$46,599.11


In [8]:
charges_by_group = defaultdict(list)

for row in records:
    group = age_and_smoking_group(row)
    charges_by_group[group].append(row["charges"])

median_gaps = {}

for band in ("18-29", "30-44", "45-59", "60+"):
    smoker_charges = charges_by_group.get((band, "yes"), [])
    nonsmoker_charges = charges_by_group.get((band, "no"), [])

    if not smoker_charges or not nonsmoker_charges:
        print(f"{band}: Both smoking groups are need for a comparison.")
        continue

    smoker_median = median(smoker_charges)
    nonsmoker_median = median(nonsmoker_charges)
    gap = smoker_median - nonsmoker_median
    median_gaps[band] = gap

    print(
        f"{band} | Smokers: ${smoker_median:,.2f} (n={len(smoker_charges)}) | "
        f"Nonsmokers: ${nonsmoker_median:,.2f} (n={len(nonsmoker_charges)}) | "
        f"Gap: ${gap:,.2f}"
    )

if median_gaps:
    print(
        f"\nMedian charge gaps ranged from ${min(median_gaps.values()):,.2f} "
        f"to ${max(median_gaps.values()):,.2f} across comparable age bands."
)

18-29 | Smokers: $33,604.25 (n=86) | Nonsmokers: $2,709.11 (n=331) | Gap: $30,895.14
30-44 | Smokers: $35,491.64 (n=91) | Nonsmokers: $6,250.44 (n=301) | Gap: $29,241.20
45-59 | Smokers: $40,847.36 (n=70) | Nonsmokers: $10,601.41 (n=345) | Gap: $30,245.95
60+ | Smokers: $46,599.11 (n=27) | Nonsmokers: $13,831.12 (n=87) | Gap: $32,767.99

Median charge gaps ranged from $29,241.20 to $32,767.99 across comparable age bands.


I analyzed 1,338 insurance records across four regions. In the full sample, median charges were \\$34,456.35 for smokers and \\$7,345.41 for nonsmokers. Among people aged 18–29, median charges were \\$33,604.25 for smokers and \\$2,709.11 for nonsmokers, a difference of \\$30,895.14. That age band included 86 smokers and 331 nonsmokers.

Across the age bands, the difference in median charges ranged from \\$29,241.20 to \\$32,767.99. These comparisons describe an association in this sample. Other characteristics, such as BMI, may also contribute to the differences.